# Machine Learning Engineering Hands-on Lab
**Five end-to-end exercises:** Linear Regression, Logistic Regression, Decision Tree, Random Forest, and K-Means.

Run from top to bottom in Google Colab. The notebook downloads public datasets automatically, evaluates models, saves CSV artifacts and creates a short PDF diagnostic report. Metrics are measured from the downloaded data; do not claim a benchmark was met unless the output confirms it.


In [ ]:
# Install/import dependencies (Colab usually already has these)
!pip -q install scikit-learn pandas numpy matplotlib seaborn reportlab


In [ ]:
import os, io, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_squared_error, r2_score, classification_report,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, f1_score,
    accuracy_score, silhouette_score, RocCurveDisplay, precision_recall_curve)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
print("Ready. Outputs will be saved in ./outputs")


## Exercise 1 — Multiple Linear Regression: Insurance Charges
Target: `charges`. Numeric features are standardized and categorical features one-hot encoded inside a pipeline to prevent leakage. We compare raw charges and log-transformed target models, then inspect residuals.


In [ ]:
insurance_url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
insurance = pd.read_csv(insurance_url)
display(insurance.head())
print("Shape:", insurance.shape, "| Charge skew:", round(insurance["charges"].skew(), 3))


In [ ]:
target = "charges"
X = insurance.drop(columns=target)
y = insurance[target]
num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
lr.fit(X_train, y_train)
pred = lr.predict(X_test)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)
print(f"Raw charges: R2={r2:.4f}, RMSE=${rmse:,.2f}")
# Compare log1p target
lr_log = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
lr_log.fit(X_train, np.log1p(y_train))
pred_log = np.expm1(lr_log.predict(X_test))
print(f"Log-target model (back-transformed): R2={r2_score(y_test, pred_log):.4f}, RMSE=${mean_squared_error(y_test, pred_log)**0.5:,.2f}")
# Smoker coefficient, recovered from fitted pipeline
feature_names = lr.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr.named_steps["model"].coef_, index=feature_names)
print("Smoker coefficient:", coef[coef.index.str.contains("smoker")].to_dict())
plt.figure(figsize=(6,4)); plt.scatter(pred, y_test-pred, alpha=.6); plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted charges ($)"); plt.ylabel("Residual ($)"); plt.title("Exercise 1: Residuals vs Fitted"); plt.tight_layout(); plt.savefig("outputs/ex1_residuals.png"); plt.show()
plt.figure(figsize=(6,4)); plt.scatter(pred, y_test, alpha=.6); lim=[min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]; plt.plot(lim,lim,"r--")
plt.xlabel("Predicted charges ($)"); plt.ylabel("Actual charges ($)"); plt.title("Exercise 1: Actual vs Predicted"); plt.tight_layout(); plt.savefig("outputs/ex1_actual_predicted.png"); plt.show()
pd.DataFrame({"actual_charges":y_test, "predicted_charges":pred, "residual":y_test-pred}).to_csv("outputs/ex1_insurance_predictions.csv", index=False)


**Interpretation prompt:** Is the residual spread larger for expensive policies? If yes, explain that non-constant variance is a limitation of the raw linear model. Log transformation may reduce skewness, but compare test metrics before choosing a model.


## Exercise 2 — Logistic Regression: Bank Marketing
Target `y` indicates whether a client subscribed. We use a stratified split, robust scaling for numeric fields, one-hot encoding for categorical fields, class weighting, ROC-AUC, a confusion matrix, and a 0.35 probability threshold.


In [ ]:
bank_zip_url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
try:
    raw = urllib.request.urlopen(bank_zip_url, timeout=30).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    inner_name = next(n for n in z.namelist() if n.endswith("bank.zip"))
    inner = zipfile.ZipFile(io.BytesIO(z.read(inner_name)))
    bank_file = next(n for n in inner.namelist() if n.endswith("bank-full.csv"))
    bank = pd.read_csv(io.BytesIO(inner.read(bank_file)), sep=";")
except Exception as e:
    print("Primary UCI download failed; trying public mirror:", e)
    mirror = "https://raw.githubusercontent.com/ Selva86/datasets/master/BankMarketing.csv".replace(" ", "")
    bank = pd.read_csv(mirror)
    if "y" not in bank.columns and "deposit" in bank.columns: bank = bank.rename(columns={"deposit":"y"})
display(bank.head())
print("Shape:", bank.shape, "Target distribution:", bank["y"].value_counts(normalize=True).round(3).to_dict())


In [ ]:
bank = bank.copy()
bank["y"] = bank["y"].astype(str).str.lower().map({"yes":1, "no":0, "1":1, "0":0})
bank = bank.dropna(subset=["y"])
Xb = bank.drop(columns=["y"])
yb = bank["y"].astype(int)
num_b = Xb.select_dtypes(include=np.number).columns.tolist()
cat_b = Xb.select_dtypes(exclude=np.number).columns.tolist()
prep_b = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", RobustScaler())]), num_b),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_b)
])
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=.25, random_state=42, stratify=yb)
logreg = Pipeline([("prep", prep_b), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42))])
logreg.fit(Xb_train, yb_train)
probs = logreg.predict_proba(Xb_test)[:,1]
threshold = .35
pred_b = (probs >= threshold).astype(int)
print("ROC-AUC:", round(roc_auc_score(yb_test, probs),4), "| threshold:", threshold)
print(classification_report(yb_test, pred_b, target_names=["No subscription","Subscribed"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(yb_test, pred_b, display_labels=["No","Yes"]); plt.title("Exercise 2: Confusion Matrix"); plt.tight_layout(); plt.savefig("outputs/ex2_confusion_matrix.png"); plt.show()
RocCurveDisplay.from_predictions(yb_test, probs); plt.title("Exercise 2: ROC Curve"); plt.tight_layout(); plt.savefig("outputs/ex2_roc.png"); plt.show()
pd.DataFrame({"actual":yb_test, "probability_subscribed":probs, "predicted_at_035":pred_b}).to_csv("outputs/ex2_bank_predictions.csv", index=False)


**Business interpretation:** A lower threshold usually catches more likely subscribers (higher recall) but can increase false-positive calls. Use the confusion matrix to explain the trade-off. The threshold of 0.35 is a lab setting, not a guarantee of 76% recall.


## Exercise 3 — Decision Tree: Interpretable Rules
We reuse the bank dataset. Compare Gini and entropy, show how an unrestricted tree can overfit, train a constrained tree, and print human-readable rules.


In [ ]:
# Reuse the preprocessed bank split; get encoded feature names
Xbt = prep_b.fit_transform(Xb_train, yb_train)
Xbv = prep_b.transform(Xb_test)
feature_names_b = prep_b.get_feature_names_out()
for criterion in ["gini", "entropy"]:
    t = DecisionTreeClassifier(criterion=criterion, random_state=42)
    t.fit(Xbt, yb_train)
    print(criterion, "| depth:", t.get_depth(), "| train acc:", round(t.score(Xbt,yb_train),4), "| test acc:", round(t.score(Xbv,yb_test),4))
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(Xbt, yb_train)
print("Unconstrained tree train/test accuracy:", round(tree_full.score(Xbt,yb_train),4), round(tree_full.score(Xbv,yb_test),4))
tree = DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
tree.fit(Xbt, yb_train)
tree_pred = tree.predict(Xbv)
print("Constrained tree | depth:", tree.get_depth(), "| test accuracy:", round(accuracy_score(yb_test,tree_pred),4), "| F1:", round(f1_score(yb_test,tree_pred),4))
print(export_text(tree, feature_names=list(feature_names_b), max_depth=3))
print("Pruning alpha candidates:", tree.cost_complexity_pruning_path(Xbt, yb_train).ccp_alphas[:10])
ConfusionMatrixDisplay.from_predictions(yb_test, tree_pred); plt.title("Exercise 3: Decision Tree"); plt.tight_layout(); plt.savefig("outputs/ex3_tree_confusion.png"); plt.show()
pd.DataFrame({"actual":yb_test, "predicted":tree_pred}).to_csv("outputs/ex3_tree_predictions.csv", index=False)


## Exercise 4 — Random Forest: Bagging and Feature Importance
Train a forest with out-of-bag validation, tune selected hyperparameters with a small grid search, and plot feature importance. The top features can differ by dataset and preprocessing.


In [ ]:
rf = RandomForestClassifier(n_estimators=200, oob_score=True, max_features="sqrt", random_state=42, n_jobs=-1, class_weight="balanced")
rf.fit(Xbt, yb_train)
rf_pred = rf.predict(Xbv)
print("OOB score:", round(rf.oob_score_,4), "| Test accuracy:", round(accuracy_score(yb_test,rf_pred),4), "| Test F1:", round(f1_score(yb_test,rf_pred),4))
grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    {"n_estimators":[100,200], "max_features":["sqrt","log2"], "max_depth":[8,12]},
    scoring="f1", cv=3, n_jobs=-1)
grid.fit(Xbt, yb_train)
print("Best parameters:", grid.best_params_, "| CV F1:", round(grid.best_score_,4))
importance = pd.Series(rf.feature_importances_, index=feature_names_b).sort_values(ascending=False)
display(importance.head(15).rename("importance").to_frame())
importance.head(15).sort_values().plot(kind="barh", figsize=(8,5), title="Exercise 4: Top Random Forest Features")
plt.tight_layout(); plt.savefig("outputs/ex4_feature_importance.png"); plt.show()
pd.DataFrame({"actual":yb_test, "predicted":rf_pred}).to_csv("outputs/ex4_forest_predictions.csv", index=False)


## Exercise 5 — K-Means: Wholesale Customer Personas
Log-transform the six annual spending categories, standardize them, compare k=2 through k=8 using inertia and silhouette score, then fit k=4 and profile clusters using original spending values.


In [ ]:
wholesale_url = "https://archive.ics.uci.edu/static/public/292/wholesale+customers.zip"
raw = urllib.request.urlopen(wholesale_url, timeout=30).read()
zw = zipfile.ZipFile(io.BytesIO(raw))
csv_name = next(n for n in zw.namelist() if n.lower().endswith(".csv"))
wholesale = pd.read_csv(io.BytesIO(zw.read(csv_name)))
# UCI dataset contains Channel and Region plus six spending fields
spend_cols = ["Fresh","Milk","Grocery","Frozen","Detergents_Paper","Delicassen"]
W = wholesale[spend_cols].copy()
W_scaled = StandardScaler().fit_transform(np.log1p(W))
ks = range(2,9)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(W_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(W_scaled, labels))
display(pd.DataFrame({"k":list(ks), "inertia":inertias, "silhouette":silhouettes}))
plt.figure(figsize=(6,4)); plt.plot(list(ks), inertias, marker="o"); plt.axvline(4,color="red",linestyle="--")
plt.xlabel("Number of clusters (k)"); plt.ylabel("Inertia"); plt.title("Exercise 5: Elbow Method"); plt.tight_layout(); plt.savefig("outputs/ex5_elbow.png"); plt.show()
plt.figure(figsize=(6,4)); plt.plot(list(ks), silhouettes, marker="o"); plt.xlabel("k"); plt.ylabel("Silhouette score"); plt.title("Exercise 5: Silhouette Analysis"); plt.tight_layout(); plt.savefig("outputs/ex5_silhouette.png"); plt.show()
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = km4.fit_predict(W_scaled)
wholesale_out = wholesale.copy()
wholesale_out["cluster"] = clusters
centroids = wholesale_out.groupby("cluster")[spend_cols].mean().round(2)
display(centroids)
print("k=4 silhouette:", round(silhouette_score(W_scaled, clusters),4))
wholesale_out.to_csv("outputs/ex5_wholesale_clusters.csv", index=False)
centroids.to_csv("outputs/ex5_cluster_spending_means.csv")


**Business interpretation:** Use each cluster's original-scale mean spending to describe purchasing patterns. Cluster numbers are arbitrary; name personas from the centroid table rather than assuming cluster 0 always represents supermarkets. Recommend inventory based on high-spend categories, and treat credit-line suggestions as hypotheses requiring business validation.


## Model Comparison and Submission Artifacts
The table below compares key test metrics. Then the notebook writes a concise PDF diagnostic brief and zips the `outputs` folder.


In [ ]:
comparison = pd.DataFrame([
    {"model":"Linear Regression", "metric":"R2 / RMSE", "score":f"R2={r2:.3f}; RMSE=${rmse:,.0f}"},
    {"model":"Logistic Regression", "metric":"ROC-AUC / Recall", "score":f"AUC={roc_auc_score(yb_test, probs):.3f}; Recall={((pred_b==1)&(yb_test==1)).sum()/max((yb_test==1).sum(),1):.3f}"},
    {"model":"Decision Tree", "metric":"Accuracy / F1", "score":f"Acc={accuracy_score(yb_test,tree_pred):.3f}; F1={f1_score(yb_test,tree_pred):.3f}; depth={tree.get_depth()}"},
    {"model":"Random Forest", "metric":"OOB / F1", "score":f"OOB={rf.oob_score_:.3f}; F1={f1_score(yb_test,rf_pred):.3f}"},
    {"model":"K-Means", "metric":"Silhouette (k=4)", "score":f"{silhouette_score(W_scaled,clusters):.3f}"}
])
display(comparison)
comparison.to_csv("outputs/model_comparison.csv", index=False)


In [ ]:
# Create a compact PDF brief (tables + text; charts are saved separately in outputs/)
styles = getSampleStyleSheet()
pdf_path = "outputs/ML_Lab_Diagnostic_Brief.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
story = [Paragraph("Machine Learning Engineering Lab — Diagnostic Brief", styles["Title"]),
         Spacer(1, 10),
         Paragraph("Summary", styles["Heading2"]),
         Paragraph("This brief reports measured test-set results from five exercises. Results are data- and split-dependent; benchmark targets are not guaranteed. See the notebook for preprocessing, plots, model settings, and detailed diagnostics.", styles["BodyText"]),
         Spacer(1, 12), Paragraph("Model comparison", styles["Heading2"])]
table_data = [list(comparison.columns)] + comparison.astype(str).values.tolist()
tbl = Table(table_data, colWidths=[115,105,270], repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.HexColor("#243247")),
    ("TEXTCOLOR",(0,0),(-1,0),colors.white),
    ("GRID",(0,0),(-1,-1),0.5,colors.grey),
    ("VALIGN",(0,0),(-1,-1),"TOP"),
    ("FONTSIZE",(0,0),(-1,-1),8),
    ("BOTTOMPADDING",(0,0),(-1,-1),6), ("TOPPADDING",(0,0),(-1,-1),6)
]))
story += [tbl, Spacer(1, 12), Paragraph("Diagnostic notes", styles["Heading2"]),
          Paragraph("1. Linear regression: inspect residuals vs fitted values for changing variance and compare raw-target and log-target RMSE/R².", styles["BodyText"]),
          Paragraph("2. Logistic regression: the probability threshold changes the precision/recall trade-off. Review the confusion matrix before interpreting outreach costs.", styles["BodyText"]),
          Paragraph("3. Decision tree: compare training and test accuracy to identify overfitting. The constrained tree is easier to explain.", styles["BodyText"]),
          Paragraph("4. Random forest: compare out-of-bag performance with test metrics and review the feature-importance plot; importance is not causation.", styles["BodyText"]),
          Paragraph("5. K-Means: use elbow and silhouette plots together, then interpret original-scale cluster means. Cluster IDs have no intrinsic meaning.", styles["BodyText"]),
          Spacer(1, 10), Paragraph("Business recommendations", styles["Heading2"]),
          Paragraph("Use the best validated model for its intended use, monitor errors on new data, and avoid treating model associations as causal conclusions. For customer segmentation, align inventory and credit policies with observed spending profiles and validate with domain experts.", styles["BodyText"])]
doc.build(story)
import shutil
shutil.make_archive("ML_Lab_Submission", "zip", root_dir="outputs")
print("Created:", pdf_path, "and ML_Lab_Submission.zip")
print("CSV artifacts:", [f for f in os.listdir("outputs") if f.endswith(".csv")])


## Final submission checklist
- Download the notebook as `.ipynb` from Colab.
- Download `ML_Lab_Submission.zip` from the Colab Files panel; it contains CSV outputs, diagnostic plots, and the PDF brief.
- Before submitting, run **Runtime → Run all** and confirm there are no errors.
- Check measured metrics against the guide's targets. If a target is missed, report it honestly and discuss likely reasons rather than editing the results.
